[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/10-autonomy-policy.ipynb)

## Colab setup

Run the next cell first on Google Colab. It sparse-checkouts this public repo and installs packages only when the kernel is Colab. Local Jupyter and VS Code skip that work.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) or in an environment cell: `os.environ["OPENAI_API_KEY"] = "sk-..."`. Do that before the lesson setup cell. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

In [ ]:
# Colab setup. Run this cell before the other code cells.
# A fresh Colab runtime cannot import tutorial.common yet, so this cell
# sparse-checkouts the repo first. Local Jupyter and VS Code skip that.
#
# An API key is required. On Google Colab, set OPENAI_API_KEY in one of these ways:
#   * Secrets (the key icon): a secret named OPENAI_API_KEY
#   * an environment cell: os.environ["OPENAI_API_KEY"] = "sk-..."
# Locally, put the key in the repository-root .env.
# Do not commit a key. This cell does not print the value.
# Edits you make in Colab stay in the session. They do not push to GitHub.

import sys
from pathlib import Path


def _on_colab():
    try:
        import google.colab
    except ImportError:
        return False
    return Path("/content").is_dir() and google.colab is not None


def _use(root):
    if not (Path(root) / "tutorial" / "common" / "colab.py").is_file():
        return False
    text = str(root)
    if text not in sys.path:
        sys.path.insert(0, text)
    return True


ready = False
if _on_colab():
    repo = Path("/content/book-agents")
    if not _use(repo):
        import shutil
        import subprocess

        if repo.exists() and not (repo / ".git").exists():
            raise RuntimeError(
                str(repo) + " exists but is not a git checkout. "
                "Move that folder aside and run this cell again."
            )
        if repo.exists():
            shutil.rmtree(repo)
        url = "https://github.com/junlinghu/book-agents.git"
        try:
            subprocess.check_call([
                "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                url, str(repo),
            ])
            subprocess.check_call([
                "git", "-C", str(repo), "sparse-checkout", "set", "tutorial",
            ])
        except subprocess.CalledProcessError:
            if repo.exists():
                shutil.rmtree(repo)
            subprocess.check_call(["git", "clone", "--depth", "1", url, str(repo)])
        if not _use(repo):
            raise RuntimeError(
                "Colab setup could not find tutorial/common/colab.py after cloning."
            )
    ready = True
else:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if _use(candidate):
            ready = True
            break
    if not ready:
        print("Not Colab. Skipped clone and pip install.")

if ready:
    from tutorial.common.colab import setup_colab

    setup_colab()


# 10. Autonomy policy

A tool call is a proposal until the harness agrees. Reads can run. A restock ticket waits. A card charge does not run.

`gate_call` returns allowed, confirm_required, or denied. Confirm uses `approval_token` from `tutorial.common.autonomy`, so a different quantity is a different token. The notebook writes an oat-milk ticket only on the second run, after you pass the token from the first run. `charge_card` stays denied. The shelf and the loop are imported above.

Shared helpers this lesson needs are imported, not copied from earlier notebooks. You can run this file by itself.

## Setup

This notebook calls the OpenAI Chat Completions API. `OPENAI_API_KEY` is required. Locally the key is loaded from the repository-root `.env` by `tutorial/common/client.py`. On Colab, the setup cell above reads a secret of the same name. The value is not printed. A missing key stops the run.

Companion notes: `10-autonomy-policy.md`.

In [ ]:
import sys
from pathlib import Path


def find_root():
    """Repo root, whether the kernel started here or in tutorial."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        marker = candidate / "tutorial" / "common" / "client.py"
        if marker.is_file():
            return candidate
    raise RuntimeError(
        "Open this notebook inside the book-agents repository. "
        "The setup cell looks for tutorial/common/client.py."
    )


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.common.client import describe_runtime
from tutorial.runtime import chat, require_key

LESSON = '10-autonomy-policy'
LESSON_NUMBER = 10
require_key()
print("tutorial:", LESSON)
print(describe_runtime())


## Shared pieces

These imports are the earlier pieces this lesson uses. They come from `tutorial.common` and `tutorial.runtime`. You do not need to run the previous notebook first.


In [ ]:
from tutorial.common.harness import check, system_text
from tutorial.common.loop import run_agent
from tutorial.common.shelf import reset_db


## Tutorial 10 — autonomy gate

Auto, confirm, and never. Cards and mail use `tutorial/common/autonomy.py`.

In [ ]:
import re

from tutorial.common.autonomy import approval_token, decide, format_decision
from tutorial.common.harness import HANDLERS, VAR, register, set_hook
from tutorial.common.shelf import gap_for, row_for

TICKETS = VAR / "tickets"

# A ticket waits. Charging and mail go through tutorial.common.autonomy.
# Every other registered tool is a read or a draft, so it is auto.
CONFIRM_TOOLS = {"write_ticket"}


def reset_tickets():
    TICKETS.mkdir(parents=True, exist_ok=True)
    for path in TICKETS.glob("*.md"):
        path.unlink()


def gate_call(name, arguments, confirmed_tokens):
    """Same three outcomes as tutorial.common.autonomy.decide: allowed, confirm_required, denied."""
    if name in {"charge_card", "send_email"}:
        expected = approval_token(name, arguments)
        token = expected if expected in confirmed_tokens else None
        return decide(name, arguments, confirmed_token=token)
    if name in CONFIRM_TOOLS:
        tier = "confirm"
    elif name in HANDLERS:
        tier = "auto"
    else:
        tier = "never"
    token = approval_token(name, arguments)
    base = {
        "tool": name,
        "tier": tier,
        "arguments": arguments,
        "approval_token": token,
        "confirmed": False,
    }
    if tier == "never":
        return {
            **base,
            "decision": "denied",
            "reason": "DENIED: " + name + " is not a tool this concierge may run.",
        }
    if tier == "confirm":
        if token in confirmed_tokens:
            return {
                **base,
                "decision": "allowed",
                "confirmed": True,
                "reason": "A person approved this exact call.",
            }
        return {
            **base,
            "decision": "confirm_required",
            "reason": "Waiting for a person to approve this exact call.",
        }
    return {
        **base,
        "decision": "allowed",
        "reason": "Auto tier. The harness ran the call without a prompt.",
    }


def write_ticket(args):
    """File one restock note. The gate must allow the call before this runs."""
    sku = str(args.get("sku", "")).strip()
    if not re.fullmatch(r"[A-Z0-9-]+", sku):
        return "ERROR: sku is not a shelf id."
    row = row_for(sku)
    if row is None:
        return "ERROR: unknown sku."
    expected = gap_for(row)
    try:
        qty = int(args.get("qty"))
    except (TypeError, ValueError):
        return "ERROR: qty must be an integer."
    if qty != expected:
        return "ERROR: qty must be the gap to par (" + str(expected) + ")."
    reason = str(args.get("reason", "")).strip() or "low stock"
    citation = str(args.get("citation", "")).strip() or "(none)"
    TICKETS.mkdir(parents=True, exist_ok=True)
    path = TICKETS / (sku + ".md")
    path.write_text(
        "\n".join([
            "# Restock ticket " + sku,
            "",
            "- sku: " + sku,
            "- name: " + row["name"],
            "- qty: " + str(qty),
            "- on_hand: " + str(row["on_hand"]),
            "- par: " + str(row["par"]),
            "- reason: " + reason,
            "- citation: " + citation,
            "- status: confirmed",
            "",
        ]),
        encoding="utf-8",
    )
    return "WROTE tickets/" + sku + ".md qty=" + str(qty)


def charge_card(args):
    del args
    return "ERROR: charge_card has no implementation. The gate should have denied it."


def send_email(args):
    del args
    return "ERROR: send_email has no implementation. The gate should have denied or held it."


register(
    "write_ticket",
    "Write a restock ticket for one sku. qty must be the shelf gap. "
    "The harness requires a person's approval token before this runs.",
    {
        "sku": {"type": "string"},
        "qty": {"type": "integer"},
        "reason": {"type": "string"},
        "citation": {"type": "string"},
    },
    ["sku", "qty"],
    write_ticket,
)
register(
    "charge_card",
    "Charge a card. This concierge never does that. The harness denies the call.",
    {"amount_cents": {"type": "integer"}},
    ["amount_cents"],
    charge_card,
)
register(
    "send_email",
    "Send email. Mail outside hearthlane.example is never sent. "
    "Mail to the counter waits for a different confirmation.",
    {
        "to": {"type": "string"},
        "body": {"type": "string"},
    },
    ["to", "body"],
    send_email,
)

set_hook("gate_call", gate_call)
set_hook("format_decision", format_decision)


## Run

`system_text()` mentions only the tools imported above. Run this cell after the ones above. Each model turn calls the Chat Completions API. A check prints a warning when the wording differs from the expected trace, and the notebook continues.

In [ ]:
reset_db()
reset_tickets()

print("=== unconfirmed ticket ===")
result = run_agent(
    "Write a restock ticket for oat milk. Do not charge a card.",
    system_text(),
)
print("ANSWER:", result["text"])
check(result["stopped"] == "final", "the model can finish while the ticket waits")
check(any("CONFIRM_REQUIRED" in line for line in result["tool_log"]), "write_ticket is confirm-tier")
check(not any(TICKETS.glob("*.md")), "no ticket file before a matching token")

tokens = set(re.findall(r"token=([0-9a-f]{8})", "\n".join(result["tool_log"])))
print("approval tokens:", ", ".join(sorted(tokens)))
check(len(tokens) == 1, "one ticket call produced one token")

print("=== charge_card stays denied, even with a token for that exact call ===")
charge_args = {"amount_cents": 100}
charge = gate_call("charge_card", charge_args, {approval_token("charge_card", charge_args)})
print(format_decision(charge))
check(charge["decision"] == "denied", "never-tier ignores the token")

print("=== same ticket, token supplied ===")
confirmed = run_agent(
    "Write a restock ticket for oat milk. Do not charge a card.",
    system_text(),
    confirmed_tokens=tokens,
)
print("ANSWER:", confirmed["text"])
written = sorted(TICKETS.glob("*.md"))
check(len(written) == 1, "one ticket file exists after confirmation")
print(written[0].read_text(encoding="utf-8"))
check("WROTE" in "\n".join(confirmed["tool_log"]), "the tool ran only after the token matched")
